<a href="https://colab.research.google.com/github/nicoavilan/Intro-Comp-Cuant-UNarino-2026/blob/main/Sesion1_Fundamentos_Computacion_Cuantica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introducción a la Computación Cuántica

**Universidad de Nariño — 2026**

Professor: [Nicolás Avilán Vargas](http://www.linkedin.com/in/nicoavilanv)

nicolasg.avilan@urosario.edu.co

**Universidad del Rosario - Escuela de Ciencias e Ingeniería**

**Matemáticas Aplicadas y Ciencias de la Computación**

## Sesión 1 — Fundamentos: qubit, superposición, medición y entrelazamiento

### Objetivos de la sesión
Al finalizar esta sesión el participante podrá:
1. Representar el estado de un qubit en notación de Dirac y en la esfera de Bloch.
2. Aplicar compuertas cuánticas de uno y dos qubits e interpretar su efecto geométrico.
3. Explicar el postulado de medición (regla de Born) y verificarlo experimentalmente en Qiskit.
4. Construir un estado entrelazado (par de Bell) y distinguirlo de una correlación clásica.

**Prerrequisitos:** álgebra lineal básica (vectores, matrices, producto interno). No se requiere experiencia previa en computación cuántica.


---
## 0. Preparación del entorno (Google Colab)

Ejecuta la siguiente celda una sola vez al inicio de la sesión. Instala Qiskit y el módulo de visualización.

In [ ]:
# Instalación (puede tardar 1-2 minutos en Colab)
%pip install qiskit qiskit-aer qiskit[visualization] --quiet

La especificación `[visualization]` instala:

* `matplotlib` (≥3.3) — backend gráfico para circuit.draw(output='mpl'), plot_histogram, plot_bloch_multivector, etc.
* `pylatexenc` (≥1.4) — necesario para circuit.draw(output='latex') / 'mpl' con notación de compuertas en LaTeX
* `seaborn` (≥0.9.0) — usado en algunas paletas de visualización de resultados
* `Pillow` (≥4.2.1) — manejo de imágenes (p. ej. exportar diagramas como PNG)
* `pydot` — grafos (usado por algunas visualizaciones de estructura de circuitos/DAG)
* `sympy` (≥1.3) — simplificación simbólica usada en ciertas rutinas de dibujo

Esta guía funciona con la versión 2.5.2 de qiskit

In [ ]:
import qiskit
print(qiskit.__version__)

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector, Operator
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_bloch_multivector, plot_histogram, array_to_latex

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline # útil para ejecutar código fuera de Colab

print("Entorno listo.")

---
## 1. El qubit y el espacio de Hilbert

### 1.1 De bit a qubit

Un bit clásico toma un valor en $\{0,1\}$. Un **qubit** es un vector unitario en el espacio de Hilbert $\mathcal{H} \cong \mathbb{C}^2$, escrito en notación de Dirac como

$$
|\psi\rangle = \alpha|0\rangle + \beta|1\rangle, \qquad \alpha,\beta \in \mathbb{C}, \qquad |\alpha|^2 + |\beta|^2 = 1,
$$

donde $|0\rangle = \begin{pmatrix}1\\0\end{pmatrix}$ y $|1\rangle = \begin{pmatrix}0\\1\end{pmatrix}$ forman la **base computacional**, ortonormal respecto al producto interno $\langle \phi|\psi\rangle$.

A diferencia del bit clásico, $|\psi\rangle$ puede existir en **superposición** de $|0\rangle$ y $|1\rangle$ simultáneamente — no es una mezcla estadística de "0 o 1 con probabilidad desconocida", sino un vector genuino en un espacio vectorial complejo.

### 1.2 Fase global vs. fase relativa

Dos estados que difieren por una **fase global** $e^{i\gamma}$, es decir $|\psi\rangle$ y $e^{i\gamma}|\psi\rangle$, son físicamente indistinguibles (no afectan ninguna probabilidad de medición).

Por el contrario, la **fase relativa** entre $\alpha$ y $\beta$ sí es observable y es la responsable de fenómenos de interferencia.

$$|\phi_\alpha\rangle = \frac{|0\rangle + e^{i\alpha}|1\rangle}{\sqrt2}, \qquad |\phi_\beta\rangle = \frac{|0\rangle + e^{i\beta}|1\rangle}{\sqrt2},$$

cuyo traslape es

$$\langle\phi_\alpha|\phi_\beta\rangle = \frac{1+e^{i(\beta-\alpha)}}{2}, \qquad |\langle\phi_\alpha|\phi_\beta\rangle| = \cos\!\left(\frac{\beta-\alpha}{2}\right).$$

### 1.3 La esfera de Bloch

Todo estado puro de un qubit (salvo fase global) puede parametrizarse como

$$
|\psi(\theta,\phi)\rangle = \cos\!\left(\frac{\theta}{2}\right)|0\rangle + e^{i\phi}\sin\!\left(\frac{\theta}{2}\right)|1\rangle, \qquad \theta\in[0,\pi],\ \phi\in[0,2\pi),
$$

que corresponde a un punto en la superficie de una esfera unitaria (esfera de Bloch), con $|0\rangle$ en el polo norte y $|1\rangle$ en el polo sur. Esta representación será nuestra herramienta visual principal en esta sesión.

## Defino y mido estados cuánticos

In [ ]:
# Statevector es una función que define un estado cuántico a partir de dos números que en general son complejos.

u = Statevector([np.sqrt(2)/2, np.sqrt(2)/2])
v = Statevector([1 / 3, 2 / 3])

print("Los vectores de estado u y v se han definido.")

In [ ]:
u.draw("latex")

In [ ]:
print(u.data)

Regla de Born

In [ ]:
print(u.probabilities())

In [ ]:
print(u.probabilities().sum())

In [ ]:
# .is_valid() verifica si la norma del vector es 1.
display(u.is_valid())
display(v.is_valid())

In [ ]:
print(u.probabilities().sum())

In [ ]:
print(v.probabilities().sum())

Al verificar que la suma de las probabilidades no es $1$ evidenciamos porqué no es un estado cuántico bien definido.

### Mediciones sobre los estados

In [ ]:
print(u.measure())

In [ ]:
print(u.measure()[0])

In [ ]:
u.measure()[1].draw("latex")

In [ ]:
u.probabilities()

In [ ]:
statistics = u.sample_counts(1000)
plot_histogram(statistics)

## Defino estados en un circuito

Hasta ahora definimos estados directamente por sus amplitudes; en la práctica los construiremos aplicando compuertas dentro de un `QuantumCircuit`.

Para definir un circuito uso la instrucción `QuantumCircuit(n,m)` el cual define n qubits y m bits clásicos.

In [ ]:
# Estado |0> por defecto en Qiskit
qc = QuantumCircuit(1)
state0 = Statevector.from_instruction(qc) # obtiene el estado cuántico en el circuito
print("Estado |0>:", state0)
state0.draw("latex")

In [ ]:
fig = plot_bloch_multivector(state0)
fig

---
## 2. Compuertas de un qubit

Las compuertas cuánticas de un solo qubit son operadores **unitarios** $U$ ($U^\dagger U = \mathbb{1}$), es decir, rotaciones (posiblemente con reflexión) sobre la esfera de Bloch. Las más usadas:

| Compuerta | Matriz | Efecto geométrico |
|---|---|---|
| $X$ | $\begin{pmatrix}0&1\\1&0\end{pmatrix}$ | Rotación $\pi$ alrededor del eje $x$ (equivalente al NOT clásico) |
| $Y$ | $\begin{pmatrix}0&-i\\i&0\end{pmatrix}$ | Rotación $\pi$ alrededor del eje $y$ |
| $Z$ | $\begin{pmatrix}1&0\\0&-1\end{pmatrix}$ | Rotación $\pi$ alrededor del eje $z$ (cambia signo de $\beta$) |
| $H$ (Hadamard) | $$\frac{1}{\sqrt2} \begin{pmatrix}1&1\\1&-1\end{pmatrix}$$ | Lleva $|0\rangle \mapsto \frac{|0\rangle+|1\rangle}{\sqrt2}$, genera superposición equitativa |
| $S$ | $\begin{pmatrix}1&0\\0&i\end{pmatrix}$ | Rotación $\pi/2$ alrededor de $z$ (fase relativa). Recordar que $i=e^{i\pi/2}$ |
| $T$ | $\begin{pmatrix}1&0\\0&e^{i\pi/4}\end{pmatrix}$ | Rotación $\pi/4$ alrededor de $z$ |

La compuerta $H$ es la que usaremos para crear superposición. Observa cómo $H|0\rangle = |+\rangle := \frac{|0\rangle+|1\rangle}{\sqrt2}$ y $H|1\rangle = |-\rangle := \frac{|0\rangle-|1\rangle}{\sqrt2}$.

Observa también que $H^2 = \mathbb{I}$.

Pregunta:
* ¿Por qué las compuertas cuánticas deben ser representadas por matrices unitarias ($U^\dagger U = \mathbb{I}$)?

In [ ]:
# Aplicamos H a |0> y visualizamos en la esfera de Bloch
qc_h = QuantumCircuit(1)
qc_h.h(0)
state_plus = Statevector.from_instruction(qc_h)
state_plus.draw("latex")

In [ ]:
Operator(qc_h).draw("latex")

In [ ]:
plot_bloch_multivector(state_plus)

In [ ]:
# Comparemos varias compuertas de un vistazo: X, H, S aplicadas sobre |0>
fig, axes = plt.subplots(1, 3, figsize=(12, 4), subplot_kw={'projection': None})
plt.close(fig)  # usaremos plot_bloch_multivector individualmente

for gate_name, apply_gate in [("X", lambda qc: qc.x(0)),
                               ("H", lambda qc: qc.h(0)),
                               ("S despues de H", lambda qc: (qc.h(0), qc.s(0)))]:
    qc_temp = QuantumCircuit(1)
    apply_gate(qc_temp)
    sv = Statevector.from_instruction(qc_temp)
    print(f"--- {gate_name} ---")
    display(array_to_latex(sv.data, prefix="|\\psi\\rangle = "))
    display(sv.draw("latex"))
    display(plot_bloch_multivector(sv))


---
## 3. Medición y regla de Born

### 3.1 El postulado de medición

Al medir $|\psi\rangle = \alpha|0\rangle+\beta|1\rangle$ en la **base computacional**, se obtiene:

$$
\text{resultado } 0 \ \text{con probabilidad } P(0) = |\alpha|^2, \qquad
\text{resultado } 1 \ \text{con probabilidad } P(1) = |\beta|^2.
$$

Este es el **postulado de Born**. Tras la medición, el estado **colapsa** al vector propio correspondiente al resultado obtenido ($|0\rangle$ o $|1\rangle$) — la información sobre $\alpha,\beta$ originales se pierde.

### 3.2 Simulación estadística: *shots*

En hardware real (y en simulación estadística), no observamos $\alpha,\beta$ directamente: debemos repetir el circuito muchas veces (*shots*) y estimar $P(0), P(1)$ a partir de las frecuencias relativas.

Distinguimos dos formas de trabajar en Qiskit:
- **`Statevector`**: simulación *exacta* del vector de estado (no hay ruido de muestreo, es matemáticamente exacto).
- **`AerSimulator` con medición y *shots***: emula lo que ocurriría en un dispositivo real, con ruido estadístico de muestreo finito.

In [ ]:
# Circuito con H y medicion, ejecutado con muchos "shots"
qc_meas = QuantumCircuit(1, 1)
qc_meas.h(0)
qc_meas.measure(0, 0) # El resultado de la medida sobre el qubit 0 es guardado en en bit 0

qc_meas.draw(output='mpl')

In [ ]:
# Simulador
sim = AerSimulator()
qc_transpiled = transpile(qc_meas, sim)
job = sim.run(qc_transpiled, shots=2000)
counts = job.result().get_counts()
print("Conteos:", counts)

plot_histogram(counts, title="Medicion de H|0> (esperado: ~50% / ~50%)")


**Observa:** las frecuencias deben acercarse a 50%/50% conforme aumentan los *shots* — es la ley de los grandes números aplicada a la regla de Born, **no** un efecto de "aleatoriedad clásica oculta". Prueba a cambiar `shots` por 10, 100 y 10000 y observa cómo se estabiliza el histograma.

---
## 4. Sistemas de varios qubits: producto tensorial

El estado de $n$ qubits vive en $\mathcal{H}^{\otimes n} \cong \mathbb{C}^{2^n}$. Para dos qubits independientes en los estados $|\psi_1\rangle$ y $|\psi_2\rangle$, el estado conjunto es el **producto tensorial**:

$$
|\psi_1\rangle \otimes |\psi_2\rangle \in \mathbb{C}^4.
$$

Por ejemplo, $|0\rangle\otimes|0\rangle \equiv |00\rangle = (1,0,0,0)^T$. En general, un estado de 2 qubits se escribe

$$
|\Psi\rangle = c_{00}|00\rangle + c_{01}|01\rangle + c_{10}|10\rangle + c_{11}|11\rangle, \qquad \sum_{ij}|c_{ij}|^2 = 1.
$$

**Punto clave:** no todo estado $|\Psi\rangle \in \mathbb{C}^4$ puede escribirse como un producto $|\psi_1\rangle\otimes|\psi_2\rangle$. Cuando esto ocurre, decimos que el estado está **entrelazado** — este es el tema de la siguiente sección, y es la razón por la cual el espacio de estados de $n$ qubits crece como $2^n$ en lugar de $2n$.

In [ ]:
# Producto tensorial de dos qubits en |0>: |00>
qc2 = QuantumCircuit(2)
state00 = Statevector.from_instruction(qc2)
display(array_to_latex(state00.data, prefix="|00\\rangle = "))

In [ ]:
# Ahora aplicamos H solo al primer qubit -> estado producto (NO entrelazado)
qc2b = QuantumCircuit(2)
qc2b.h(0)
state_prod = Statevector.from_instruction(qc2b)
display(array_to_latex(state_prod.data, prefix="(H\\otimes \\mathbb{1})|00\\rangle = "))

---
## 5. Entrelazamiento: estados de Bell

### 5.1 Definición

Un estado de dos qubits $|\Psi\rangle$ está **entrelazado** si no existe factorización $|\Psi\rangle = |\psi_1\rangle\otimes|\psi_2\rangle$. El ejemplo canónico es el estado de Bell

$$
|\Phi^+\rangle = \frac{1}{\sqrt2}\big(|00\rangle + |11\rangle\big).
$$

Este estado **no puede** escribirse como producto de dos estados individuales (inténtalo: no hay $\alpha,\beta,\gamma,\delta$ tales que $(\alpha|0\rangle+\beta|1\rangle)\otimes(\gamma|0\rangle+\delta|1\rangle)$ reproduzca los coeficientes $\frac{1}{\sqrt2}, 0, 0, \frac{1}{\sqrt2}$, ya que exigiría $\beta\gamma=0$ y $\alpha\delta = 0$ simultáneamente con $\alpha\gamma=\beta\delta=\frac1{\sqrt2}\neq0$).

Los cuatro estados de Bell forman una base ortonormal del espacio de 2 qubits:

$$
|\Phi^\pm\rangle = \frac{|00\rangle \pm |11\rangle}{\sqrt2}, \qquad
|\Psi^\pm\rangle = \frac{|01\rangle \pm |10\rangle}{\sqrt2}.
$$

### 5.2 Circuito generador: H + CNOT

El circuito estándar para preparar $|\Phi^+\rangle$ a partir de $|00\rangle$ es:
1. Aplicar $H$ al primer qubit: $|00\rangle \mapsto \frac{|00\rangle+|10\rangle}{\sqrt2}$
2. Aplicar $\text{CNOT}$ (control = qubit 0, objetivo = qubit 1): $\frac{|00\rangle+|10\rangle}{\sqrt2} \mapsto \frac{|00\rangle+|11\rangle}{\sqrt2}$

La compuerta CNOT actúa como $|c\rangle|t\rangle \mapsto |c\rangle|t\oplus c\rangle$ (XOR clásico condicionado al qubit de control).

In [ ]:
# Construccion del estado de Bell |Phi+>
qc_bell = QuantumCircuit(2)
qc_bell.h(0)
qc_bell.cx(0, 1)   # CNOT: control=0, target=1

qc_bell.draw(output='mpl')

In [ ]:
state_bell = Statevector.from_instruction(qc_bell)
display(array_to_latex(state_bell.data, prefix="|\\Phi^+\\rangle = "))

In [ ]:
# Verificacion experimental: medimos muchas veces y solo deberian aparecer '00' y '11'
qc_bell_meas = QuantumCircuit(2, 2)
qc_bell_meas.h(0)
qc_bell_meas.cx(0, 1)
qc_bell_meas.measure([0, 1], [0, 1])

sim = AerSimulator()
qc_t = transpile(qc_bell_meas, sim)
counts = sim.run(qc_t, shots=2000).result().get_counts()
print("Conteos:", counts)
plot_histogram(counts, title="Medicion del estado de Bell (esperado: solo '00' y '11')")


**Interpretación física:** al medir el primer qubit, el resultado es aleatorio (50/50), pero **una vez conocido**, el segundo qubit queda determinado con certeza — sin importar la distancia física entre ambos. Este es el núcleo de la correlación cuántica que Einstein, Podolsky y Rosen (EPR) discutieron en 1935, y que no tiene análogo en correlaciones clásicas locales (violación de las desigualdades de Bell, tema que puedes explorar por tu cuenta como profundización).

---
## 6. Compuertas de dos qubits

| Compuerta | Acción | Matriz $$(\text{base } |00\rangle,|01\rangle,|10\rangle,|11\rangle)$$ |
|---|---|---|
| CNOT ($CX$) | $$|c\rangle |t\rangle \mapsto |c\rangle |t\oplus c\rangle$$ | $\begin{pmatrix}1&0&0&0\\0&1&0&0\\0&0&0&1\\0&0&1&0\end{pmatrix}$ |
| CZ | $|c\rangle|t\rangle \mapsto (-1)^{c\cdot t}|c\rangle|t\rangle$ | $$\text{diag}(1,1,1,-1)$$ |

Ambas son necesarias (junto con compuertas de 1 qubit) para generar entrelazamiento — ninguna compuerta de un solo qubit, actuando independientemente sobre cada subsistema, puede entrelazar un estado producto.

In [ ]:
qc_cnot = QuantumCircuit(2)
Statevector.from_instruction(qc_cnot).draw("latex")

In [ ]:
qc_cnot.h(0)
Statevector.from_instruction(qc_cnot).draw("latex")

`cx(control,target)`

In [ ]:
qc_cnot.cx(0,1)
Statevector.from_instruction(qc_cnot).draw("latex")

Obtuvimos el primer estado de Bell, que es un estado entrelazado.

---
## 7. Resumen de la sesión

| Concepto | Idea clave |
|---|---|
| Qubit | Vector unitario en $\mathbb{C}^2$; superposición $\alpha|0\rangle+\beta|1\rangle$ |
| Esfera de Bloch | Representación geométrica de estados puros de 1 qubit |
| Medición | Regla de Born: $P(k)=|c_k|^2$; colapso del estado |
| Multi-qubit | Espacio $\mathbb{C}^{2^n}$ vía producto tensorial |
| Entrelazamiento | Estado no factorizable; ejemplo: pares de Bell |
| Compuertas | Operadores unitarios; $H$+CNOT generan entrelazamiento |


---
## 8. Ejercicios propuestos

Resuelve los siguientes ejercicios en las celdas de código provistas.


### Ejercicio 1 — Estados y probabilidades (teórico-práctico)
Dado el estado $\frac{\sqrt 3}{2}|0\rangle + \frac{1}{2}|1\rangle $:

* (a) Verifica a mano que está normalizado.
* (b) Constrúyelo en Qiskit usando `Statevector` o `QuantumCircuit`, calcula $P(0)$ y $P(1)$ con `probabilities()`.
* (c) Verifica con experimentos el resultado anterior.
* (d) Grafica el estado en la esfera de Bloch.


In [ ]:
# Ejercicio 1 - tu codigo aqui
import numpy as np

# (b) Construye el estado (pista: usa QuantumCircuit(1))




### Ejercicio 2 — Estados y probabilidades (teórico-práctico)
Dado el estado $|\psi\rangle = \frac{1}{\sqrt3}|0\rangle + \sqrt{\frac{2}{3}}\,e^{i\pi/4}|1\rangle$:

* (a) Verifica a mano que está normalizado.
* (b) Constrúyelo en Qiskit usando `Statevector` o `QuantumCircuit`, calcula $P(0)$ y $P(1)$ con `probabilities()`.
* (c) Verifica con experimentos el resultado anterior.
* (d) Grafica el estado en la esfera de Bloch.

In [ ]:
# Ejercicio 2 - tu codigo aqui
import numpy as np

# (b) Construye el estado (pista: usa QuantumCircuit(1))




### Ejercicio 3 — Secuencias de compuertas
Sin ejecutar código, predice el estado final (en notación de Dirac) tras aplicar, en orden, $H$, luego $Z$, luego $H$ sobre $|0\rangle$. Verifica tu predicción con Qiskit y explica el resultado en términos de interferencia.

In [ ]:
# Ejercicio 3 - verifica tu prediccion aqui
qc_ex3 = QuantumCircuit(1)
# completa: qc_ex3.h(0); qc_ex3.z(0); qc_ex3.h(0)




### Ejercicio 4 — Los cuatro estados de Bell
Construye los circuitos para preparar
* $|\Phi^-\rangle = \frac{1}{\sqrt 2}|00\rangle - \frac{1}{\sqrt 2}|11\rangle $
* $|\Psi^+\rangle = \frac{1}{\sqrt 2}|01\rangle + \frac{1}{\sqrt 2}|10\rangle $
* $|\Psi^-\rangle = \frac{1}{\sqrt 2}|01\rangle - \frac{1}{\sqrt 2}|10\rangle $


(pista: parte del circuito de $|\Phi^+\rangle$ y agrega una compuerta $X$ y/o $Z$ *antes* del CNOT, sobre el qubit adecuado). Verifica cada uno con `Statevector` y con histogramas de medición.

In [ ]:
# Ejercicio 4 - tu codigo aqui






### Ejercicio 5 — Reto (opcional)
Investiga y construye en Qiskit el circuito del **estado GHZ de 3 qubits**, $|\text{GHZ}\rangle = \frac{|000\rangle+|111\rangle}{\sqrt2}$. Mide los 3 qubits 2000 veces y verifica que solo aparecen los resultados `000` y `111`. ¿Cómo generalizarías el patrón H+CNOT usado para el par de Bell?

In [ ]:
# Ejercicio 5 (reto) - tu codigo aqui
qc_ghz = QuantumCircuit(3, 3)
# completa el circuito




---
## Referencias para profundizar
- IBM. ¿Qué es computación cuántica? IBM Think. https://www.ibm.com/mx-es/think/topics/quantum-computing (consultado en 2026).
- Nielsen, M. A. & Chuang, I. L. *Quantum Computation and Quantum Information*. Cambridge University Press.
- Basics of quantum information: https://quantum.cloud.ibm.com/learning/en/courses/basics-of-quantum-information
- Qiskit Textbook / Documentation: https://docs.quantum.ibm.com/

* Dür, W. & Heusler, S. (2013). What we can learn about quantum physics from a single qubit. arXiv. [arxiv.org/pdf/1312.1463](https://arxiv.org/pdf/1312.1463)


Para informar de errores o hacer sugerencias: nicolasg.avilan@urosario.edu.co